# Análisis de CPUs: ingesta y calidad de datos

## Objetivo
Preparar un conjunto de datos confiables sobre procesadores comerciales para analizar rendimineto, precio, eficiencia energetica, pltaforma y categorías mediante python, sql y PowerBI



## Hipotesis
 Los procesadores registrados en años más recientes tienden a presentar mayor rendimiento (cpuMark) y mayor eficiencia energética (powerPerf). Sin embargo, se evaluará si ese incremento de rendimiento se traduce también en precios más altos y si existen categorías o plataformas que ofrecen una mejor relación rendimiento/precio.
 
Se contestarán las siguinetes preguntas:
¿Qué procesadores ofrecen mejor rendimiento por dólar?
¿Cómo se relacionan precio, rendimiento, núcleos y TDP?
¿Qué categorías y sockets concentran más modelos?
¿Qué problemas de calidad existen en la fuente?
¿Qué variables serían válidas para estimar precio?

## Diccionario de datos
cpuName: Nombre del microprocesador 
price: Precio del microprocesador
cpuMark: Puntuación numérica estandarizada que mide la potencia y el rendimiento de un procesador (CPU)
cpuValue:  Rendimiento por dólar
threadMark: Métrica de rendimiento
threadValue: Rendimiento de hilos por dólar
TDP: Potencia térmica de diseño, expresada en watts
powerPerf: Rendimiento cpuMark por watt de TDP
cores: Nucleos fisicos de un procresador
testDate: Año de prueba
socket: Zócalo o plataforma de compatibilidad del procesador
category: Tipo de Plataforma

In [1]:
# Librerias
import pandas as pd

In [2]:
# lectura de datos
df= pd.read_csv("../data/raw/CPU_benchmark_v4.csv")

In [3]:
# Observamos la estructura del dataframe y los tipos de datos de cada columna
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3825 entries, 0 to 3824
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cpuName      3825 non-null   object 
 1   price        1967 non-null   float64
 2   cpuMark      3825 non-null   int64  
 3   cpuValue     1967 non-null   float64
 4   threadMark   3825 non-null   int64  
 5   threadValue  1967 non-null   float64
 6   TDP          3140 non-null   float64
 7   powerPerf    3140 non-null   object 
 8   cores        3825 non-null   int64  
 9   testDate     3825 non-null   int64  
 10  socket       3825 non-null   object 
 11  category     3825 non-null   object 
dtypes: float64(4), int64(4), object(4)
memory usage: 358.7+ KB


In [4]:
#Observamos las primeras 10 filas del dataframe
df.head(10)

,cpuName,price,cpuMark,cpuValue,threadMark,threadValue,TDP,powerPerf,cores,testDate,socket,category
0,AMD Ryzen Threadripper PRO 5995WX,NaN,108822,NaN,3330,NaN,280.0,388.65,64,2022,sWRX8,Desktop
1,AMD EPYC 7763,7299.99,88338,12.10,2635,0.36,280.0,315.49,64,2021,SP3,Server
2,AMD EPYC 7J13,NaN,86006,NaN,2387,NaN,NaN,NaN,64,2021,unknown,Server
3,AMD EPYC 7713,7060.00,85861,12.16,2727,0.39,225.0,381.6,64,2021,SP3,Server
4,AMD Ryzen Threadripper PRO 3995WX,6807.98,83971,12.33,2626,0.39,280.0,299.9,64,2020,sWRX8,Desktop
5,AMD Ryzen Threadripper 3990X,8399.69,81568,9.71,2569,0.31,280.0,291.31,64,2020,sTRX4,Desktop
6,AMD Ryzen Threadripper PRO 5975WX,NaN,80842,NaN,3340,NaN,280.0,288.72,32,2022,sWRX8,Desktop
7,AMD EPYC 7B13,NaN,77460,NaN,2564,NaN,NaN,NaN,60,2021,unknown,Server
8,AMD EPYC 7643,5424.99,76455,14.09,2695,0.50,225.0,339.8,48,2021,SP3,Server
9,AMD EPYC 7702,4000.00,71646,17.91,2097,0.52,200.0,358.23,64,2020,SP3,Server


In [5]:
#ordenar los datos por la columna powerPerf de manera ascendente y mostrar las primeras 10 filas del dataframe ordenado
df_ordenado = df.sort_values(by='powerPerf', ascending=True).copy()

    

In [6]:
#Se muentran las primeras 10 filas del dataframe ordenado de manera ascendente por la columna powerPerf
df_ordenado.head(10)

,cpuName,price,cpuMark,cpuValue,threadMark,threadValue,TDP,powerPerf,cores,testDate,socket,category
430,AMD Ryzen 5 5625U,NaN,15099,NaN,3050,NaN,15.0,"1,006.60",6,2022,FP6,Laptop
425,AMD Ryzen 5 5600U,NaN,15241,NaN,2924,NaN,15.0,"1,016.06",6,2021,FP6,Laptop
418,AMD Ryzen 5 PRO 5650U,NaN,15373,NaN,2965,NaN,15.0,"1,024.86",6,2021,FP6,Laptop
412,AMD Ryzen 7 PRO 4750U,NaN,15511,NaN,2562,NaN,15.0,"1,034.04",8,2020,FP6,Laptop
400,AMD Ryzen 7 4850U Mobile,NaN,15920,NaN,2653,NaN,15.0,"1,061.35",8,2022,FP6,Laptop
396,AMD Ryzen 7 5700U,NaN,16104,NaN,2618,NaN,15.0,"1,073.63",8,2021,FP6,Laptop
374,AMD Ryzen 7 4800U,NaN,17073,NaN,2603,NaN,15.0,"1,138.18",8,2020,FP6,Laptop
358,AMD Ryzen 7 PRO 5850U,NaN,17581,NaN,3057,NaN,15.0,"1,172.06",8,2021,FP6,Laptop
346,AMD Ryzen Embedded V2718,NaN,17972,NaN,2439,NaN,15.0,"1,198.15",8,2021,FP6,Mobile/Embedded
343,AMD Ryzen 7 5825U,NaN,18122,NaN,3064,NaN,15.0,"1,208.12",8,2022,AM4,Laptop


La columna powerPerf fue cargada como tipo object porque contiene valores con comas como separadores de miles, por ejemplo 1,006.60. Debido a que no todos los registros tienen un formato numérico compatible directamente con float, pandas interpreta la columna como texto. Para utilizarla en cálculos y visualizaciones, se eliminarán las comas de miles y se convertirá la columna a tipo numérico decimal.

In [7]:
#Se limpian la columna de powerPerf de las comas y se convierte a tipo float
df_ordenado['powerPerf'] = df_ordenado['powerPerf'].str.replace(',', '').astype(float)


In [8]:
#Muestramos la información del dataframe ordenado para verificar los tipos de datos y la cantidad de valores nulos, observamos que la columna powerPerf ya es de tipo float 
df_ordenado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3825 entries, 430 to 3817
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cpuName      3825 non-null   object 
 1   price        1967 non-null   float64
 2   cpuMark      3825 non-null   int64  
 3   cpuValue     1967 non-null   float64
 4   threadMark   3825 non-null   int64  
 5   threadValue  1967 non-null   float64
 6   TDP          3140 non-null   float64
 7   powerPerf    3140 non-null   float64
 8   cores        3825 non-null   int64  
 9   testDate     3825 non-null   int64  
 10  socket       3825 non-null   object 
 11  category     3825 non-null   object 
dtypes: float64(5), int64(4), object(3)
memory usage: 388.5+ KB


## Validación de la métrica powerPerf

Se plantea la hipótesis de que `powerPerf` representa la relación entre
`cpuMark` y `TDP`, es decir, rendimiento por watt de potencia térmica de diseño.
Esta relación se validará comparando el valor reportado de `powerPerf` con el
resultado de dividir `cpuMark` entre `TDP`.

In [9]:
df_ordenado.insert(8, column="validacionTDP", value=df_ordenado["cpuMark"]/df_ordenado["TDP"])

In [15]:
df_ordenado.head(10)

,cpuName,price,cpuMark,cpuValue,threadMark,threadValue,TDP,powerPerf,validacionTDP,cores,testDate,socket,category
430,AMD Ryzen 5 5625U,NaN,15099,NaN,3050,NaN,15.0,1006.60,1006.600000,6,2022,FP6,Laptop
425,AMD Ryzen 5 5600U,NaN,15241,NaN,2924,NaN,15.0,1016.06,1016.066667,6,2021,FP6,Laptop
418,AMD Ryzen 5 PRO 5650U,NaN,15373,NaN,2965,NaN,15.0,1024.86,1024.866667,6,2021,FP6,Laptop
412,AMD Ryzen 7 PRO 4750U,NaN,15511,NaN,2562,NaN,15.0,1034.04,1034.066667,8,2020,FP6,Laptop
400,AMD Ryzen 7 4850U Mobile,NaN,15920,NaN,2653,NaN,15.0,1061.35,1061.333333,8,2022,FP6,Laptop
396,AMD Ryzen 7 5700U,NaN,16104,NaN,2618,NaN,15.0,1073.63,1073.600000,8,2021,FP6,Laptop
374,AMD Ryzen 7 4800U,NaN,17073,NaN,2603,NaN,15.0,1138.18,1138.200000,8,2020,FP6,Laptop
358,AMD Ryzen 7 PRO 5850U,NaN,17581,NaN,3057,NaN,15.0,1172.06,1172.066667,8,2021,FP6,Laptop
346,AMD Ryzen Embedded V2718,NaN,17972,NaN,2439,NaN,15.0,1198.15,1198.133333,8,2021,FP6,Mobile/Embedded
343,AMD Ryzen 7 5825U,NaN,18122,NaN,3064,NaN,15.0,1208.12,1208.133333,8,2022,AM4,Laptop


In [11]:
buscar=df_ordenado[df_ordenado["validacionTDP"].isnull()]
print(buscar)

                                   cpuName  price  cpuMark  cpuValue  \
2                            AMD EPYC 7J13    NaN    86006       NaN   
7                            AMD EPYC 7B13    NaN    77460       NaN   
16                           AMD EPYC 7R32    NaN    64727       NaN   
57    Intel Xeon Platinum 8275CL @ 3.00GHz    NaN    40794       NaN   
68        ARM Neoverse-N1 80 Core 3000 MHz    NaN    38122       NaN   
...                                    ...    ...      ...       ...   
3810                 Intel Celeron 1.70GHz    NaN      101       NaN   
3811                          VIA Nehemiah    NaN      101       NaN   
3813       Intel Pentium III Mobile 866MHz    NaN       99       NaN   
3815                    VIA Esther 1000MHz    NaN       95       NaN   
3817          ARM Cortex-A7 2 Core 960 MHz    NaN       93       NaN   

      threadMark  threadValue  TDP  powerPerf  validacionTDP  cores  testDate  \
2           2387          NaN  NaN        NaN         

In [12]:
#Vemos el tipo de dato de la columna powerPerf para confirmar que es float
df_ordenado['powerPerf'].dtype

dtype('float64')

In [13]:
#Vemos la descripción estadística de la columna powerPerf para obtener información sobre su distribución, incluyendo el conteo, media, desviación estándar, valores mínimo y máximo, y los percentiles.
df_ordenado['powerPerf'].describe()

count    3140.000000
mean      121.623239
std       151.873884
min         1.350000
25%        25.807500
50%        69.885000
75%       153.675000
max      1357.930000
Name: powerPerf, dtype: float64

In [14]:
#Valores nulos en la columna powerPerf
df_ordenado['powerPerf'].isna().sum()


np.int64(685)

In [17]:
15099//15

1006